# Exploration — Détection de Fraude Bancaire

**Auteur**: Anthony  
**Contexte**: Hackathon IBM x EFREI  

Ce notebook couvre :
1. Setup & chargement des données
2. Vue globale des données (shape, types, missing values, imbalance)
3. Insights transactions (montants, temporel, use_chip, errors, géo)
4. Enrichissement cartes (card_on_dark_web, chip mismatch, expiration)
5. Enrichissement utilisateurs (profils, distance domicile-transaction)
6. Analyse MCC (catégories marchandes à risque)
7. Feature Engineering recommandé
8. Modèles conseillés

---
**Point critique**: les clients de l'ensemble d'évaluation sont **différents** de ceux de l'entraînement.  
Toutes les features basées sur l'historique d'un client spécifique sont du **leakage** — les features retenues doivent généraliser à des clients inconnus.

## Section 1 — Setup & Chargement des données

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

DATA = Path('../dataset')

In [ ]:
# --- Transactions
txn = pd.read_csv(DATA / 'transactions_train.csv')
txn['amount'] = txn['amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
txn['date'] = pd.to_datetime(txn['date'])

# --- Labels
with open(DATA / 'train_fraud_labels.json') as f:
    raw_labels = json.load(f)
labels = pd.DataFrame({'transaction_id': list(raw_labels.keys()), 'fraud_label': list(raw_labels.values())})
labels['transaction_id'] = labels['transaction_id'].astype(int)

# --- Users
users = pd.read_csv(DATA / 'users_data.csv')
for col in ['per_capita_income', 'yearly_income', 'total_debt']:
    users[col] = users[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# --- Cards
cards = pd.read_csv(DATA / 'cards_data.csv')
cards['credit_limit'] = cards['credit_limit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# --- MCC
with open(DATA / 'mcc_codes.json') as f:
    mcc_codes = json.load(f)
mcc_df = pd.DataFrame(list(mcc_codes.items()), columns=['mcc', 'mcc_description'])
mcc_df['mcc'] = mcc_df['mcc'].astype(int)

# --- Dataset principal : merge transactions + labels
df = txn.merge(labels, on='transaction_id', how='left')
df['fraud_label'] = df['fraud_label'].fillna(0).astype(int)

print(f'Transactions: {len(txn):,}')
print(f'Labels disponibles: {len(labels):,}')
print(f'Users: {len(users):,}')
print(f'Cards: {len(cards):,}')
print(f'MCC codes: {len(mcc_df):,}')

## Section 2 — Vue Globale des Données

In [ ]:
# --- Shape et types
print('=== TRANSACTIONS ===')
print(df.dtypes)
print('\n=== USERS ===')
print(users.dtypes)
print('\n=== CARDS ===')
print(cards.dtypes)

In [ ]:
# --- Valeurs manquantes
def missing_report(data, name):
    miss = data.isnull().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    pct = (miss / len(data) * 100).round(2)
    report = pd.DataFrame({'missing': miss, 'pct_%': pct})
    print(f'\n=== {name} — valeurs manquantes ===')
    print(report if len(report) else '  Aucune valeur manquante.')

missing_report(df, 'Transactions')
missing_report(users, 'Users')
missing_report(cards, 'Cards')

In [ ]:
# --- Déséquilibre de classes
fraud_counts = df['fraud_label'].value_counts()
fraud_pct = df['fraud_label'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Non-fraude (0)', 'Fraude (1)'], fraud_counts.values, color=['#4C72B0', '#DD8452'])
axes[0].set_title(f'Distribution du target (fraude = {fraud_pct:.3f}%)')
axes[0].set_ylabel('Nombre de transactions')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(fraud_counts.values, labels=['Non-fraude', 'Fraude'],
            autopct='%1.3f%%', colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1].set_title('Proportion fraude vs non-fraude')

plt.tight_layout()
plt.show()

print(f'\nTransactions totales : {len(df):,}')
print(f'Fraudes             : {fraud_counts[1]:,} ({fraud_pct:.3f}%)')
print(f'Non-fraudes         : {fraud_counts[0]:,}')
print(f'Ratio imbalance     : 1 fraude pour {fraud_counts[0]//fraud_counts[1]:.0f} non-fraudes')

## Section 3 — Insights Transactions

In [ ]:
# --- Distribution des montants : fraude vs non-fraude (log scale)
fraud = df[df['fraud_label'] == 1]['amount']
legit = df[df['fraud_label'] == 0]['amount']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(legit.clip(lower=0), bins=80, alpha=0.6, label='Non-fraude', color='#4C72B0', log=True)
axes[0].hist(fraud.clip(lower=0), bins=80, alpha=0.8, label='Fraude', color='#DD8452', log=True)
axes[0].set_title('Distribution des montants (log scale)')
axes[0].set_xlabel('Montant ($)')
axes[0].set_ylabel('Count (log)')
axes[0].legend()

axes[1].boxplot([legit.clip(-200, 2000), fraud.clip(-200, 2000)], labels=['Non-fraude', 'Fraude'])
axes[1].set_title('Boxplot montants (-200$ à 2000$)')
axes[1].set_ylabel('Montant ($)')

plt.tight_layout()
plt.show()

print('\nStats montants — Non-fraude:')
print(legit.describe().round(2))
print('\nStats montants — Fraude:')
print(fraud.describe().round(2))

In [ ]:
# --- Montants négatifs
neg_all = (df['amount'] < 0).sum()
neg_fraud = (df[df['fraud_label'] == 1]['amount'] < 0).sum()
print(f'Montants négatifs total   : {neg_all:,} ({neg_all/len(df)*100:.2f}%)')
print(f'Montants négatifs (fraude): {neg_fraud:,} ({neg_fraud/fraud_counts[1]*100:.2f}% des fraudes)')

In [ ]:
# --- Taux de fraude par type de transaction (use_chip)
chip_fraud = df.groupby('use_chip')['fraud_label'].agg(['mean', 'sum', 'count']).rename(
    columns={'mean': 'fraud_rate', 'sum': 'fraud_count', 'count': 'total'})
chip_fraud['fraud_rate_pct'] = (chip_fraud['fraud_rate'] * 100).round(3)
print(chip_fraud.sort_values('fraud_rate', ascending=False))

fig, ax = plt.subplots(figsize=(9, 4))
chip_fraud['fraud_rate_pct'].sort_values(ascending=False).plot.bar(ax=ax, color=['#DD8452','#4C72B0','#55A868'])
ax.set_title('Taux de fraude par type de transaction')
ax.set_ylabel('Taux de fraude (%)')
ax.set_xlabel('')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# --- Patterns temporels
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek  # 0=Lundi, 6=Dimanche
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Taux de fraude par heure
hourly = df.groupby('hour')['fraud_label'].mean() * 100
axes[0].plot(hourly.index, hourly.values, marker='o', color='#DD8452')
axes[0].set_title('Taux de fraude par heure de la journée')
axes[0].set_xlabel('Heure')
axes[0].set_ylabel('Taux de fraude (%)')
axes[0].axvspan(0, 6, alpha=0.1, color='navy', label='Nuit (0h-6h)')
axes[0].legend()

# Par jour de la semaine
day_names = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
daily = df.groupby('day_of_week')['fraud_label'].mean() * 100
axes[1].bar(day_names, daily.values, color='#4C72B0')
axes[1].set_title('Taux de fraude par jour de la semaine')
axes[1].set_ylabel('Taux de fraude (%)')

# Par mois
monthly = df.groupby('month')['fraud_label'].mean() * 100
axes[2].bar(monthly.index, monthly.values, color='#55A868')
axes[2].set_title('Taux de fraude par mois')
axes[2].set_xlabel('Mois')
axes[2].set_ylabel('Taux de fraude (%)')

plt.tight_layout()
plt.show()

In [ ]:
# --- Colonne errors : corrélation avec fraude
df['has_error'] = df['errors'].notna().astype(int)

error_fraud = df.groupby('has_error')['fraud_label'].agg(['mean', 'sum', 'count'])
error_fraud.index = ['Pas d\'erreur', 'A une erreur']
error_fraud['fraud_rate_pct'] = (error_fraud['mean'] * 100).round(3)
print(error_fraud)

if df['has_error'].sum() > 0:
    top_errors = df[df['errors'].notna()]['errors'].value_counts().head(10)
    print('\nTop types d\'erreurs:')
    print(top_errors)

In [ ]:
# --- Transactions online vs physiques, et étrangères
df['is_online'] = (df['merchant_city'] == 'ONLINE').astype(int)
df['is_foreign'] = (~df['merchant_state'].isin(
    ['AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA',
     'ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK',
     'OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','DC','PR',
     float('nan')])).astype(int)
# Les transactions ONLINE n'ont pas de merchant_state
df.loc[df['is_online'] == 1, 'is_foreign'] = 0

geo_summary = pd.DataFrame({
    'Total': [df['is_online'].sum(), df['is_foreign'].sum()],
    'Fraudes': [df[df['is_online']==1]['fraud_label'].sum(), df[df['is_foreign']==1]['fraud_label'].sum()],
    'Taux fraude %': [
        df[df['is_online']==1]['fraud_label'].mean()*100,
        df[df['is_foreign']==1]['fraud_label'].mean()*100
    ]
}, index=['Online', 'Étrangère'])
geo_summary['Taux fraude %'] = geo_summary['Taux fraude %'].round(3)
print(geo_summary)

## Section 4 — Enrichissement Cartes (`cards_data`)

In [ ]:
# Merge transactions enrichi avec cartes
df_cards = df.merge(cards.rename(columns={'id': 'card_id'}), on='card_id', how='left')
print(f'Lignes après merge cards: {len(df_cards):,} (attendu: {len(df):,})')
print(f'Couverture merge: {df_cards["card_brand"].notna().mean()*100:.1f}%')

In [ ]:
# --- card_on_dark_web : signal fort
if 'card_on_dark_web' in df_cards.columns:
    dw = df_cards.groupby('card_on_dark_web')['fraud_label'].agg(['mean', 'sum', 'count'])
    dw['fraud_rate_pct'] = (dw['mean'] * 100).round(3)
    print('=== card_on_dark_web vs taux de fraude ===')
    print(dw)

    fig, ax = plt.subplots(figsize=(7, 4))
    dw['fraud_rate_pct'].plot.bar(ax=ax, color=['#4C72B0', '#DD8452'])
    ax.set_title('Taux de fraude : carte présente sur le dark web ?')
    ax.set_ylabel('Taux de fraude (%)')
    ax.set_xlabel('card_on_dark_web')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Chip mismatch : carte avec chip mais transaction en Swipe
df_cards['chip_mismatch'] = (
    (df_cards['has_chip'] == 'YES') & (df_cards['use_chip'] == 'Swipe Transaction')
).astype(int)

mismatch_fraud = df_cards.groupby('chip_mismatch')['fraud_label'].agg(['mean','sum','count'])
mismatch_fraud.index = ['Pas de mismatch', 'Chip mismatch (swipe sur chip card)']
mismatch_fraud['fraud_rate_pct'] = (mismatch_fraud['mean'] * 100).round(3)
print('=== Chip mismatch vs taux de fraude ===')
print(mismatch_fraud)

In [ ]:
# --- Carte expirée au moment de la transaction
df_cards['card_expiry_date'] = pd.to_datetime(df_cards['expires'], format='%m/%Y', errors='coerce')
df_cards['card_expired_at_txn'] = (df_cards['date'] > df_cards['card_expiry_date']).astype(int)

expired_fraud = df_cards.groupby('card_expired_at_txn')['fraud_label'].agg(['mean','sum','count'])
expired_fraud.index = ['Carte valide', 'Carte expirée']
expired_fraud['fraud_rate_pct'] = (expired_fraud['mean'] * 100).round(3)
print('=== Carte expirée vs taux de fraude ===')
print(expired_fraud)

In [ ]:
# --- Ratio montant / credit_limit
df_cards['amount_to_limit_ratio'] = df_cards['amount'] / (df_cards['credit_limit'] + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_cards[df_cards['fraud_label']==0]['amount_to_limit_ratio'].clip(-0.1, 2),
        bins=60, alpha=0.6, label='Non-fraude', color='#4C72B0', log=True)
ax.hist(df_cards[df_cards['fraud_label']==1]['amount_to_limit_ratio'].clip(-0.1, 2),
        bins=60, alpha=0.8, label='Fraude', color='#DD8452', log=True)
ax.set_title('Ratio montant / credit_limit (log scale)')
ax.set_xlabel('amount / credit_limit')
ax.set_ylabel('Count (log)')
ax.legend()
plt.tight_layout()
plt.show()

## Section 5 — Enrichissement Utilisateurs (`users_data`)

In [ ]:
# Merge avec users via client_id
df_full = df_cards.merge(users.rename(columns={'id': 'client_id'}), on='client_id', how='left')
print(f'Couverture users: {df_full["yearly_income"].notna().mean()*100:.1f}%')

In [ ]:
# --- Profils financiers : fraudés vs non-fraudés
financial_cols = ['credit_score', 'yearly_income', 'total_debt', 'current_age']
available = [c for c in financial_cols if c in df_full.columns]

fig, axes = plt.subplots(1, len(available), figsize=(5*len(available), 4))
if len(available) == 1:
    axes = [axes]

for ax, col in zip(axes, available):
    fraud_vals = df_full[df_full['fraud_label']==1][col].dropna()
    legit_vals = df_full[df_full['fraud_label']==0][col].dropna()
    ax.boxplot([legit_vals, fraud_vals], labels=['Non-fraude', 'Fraude'])
    ax.set_title(col)
    ax.set_ylabel(col)

plt.suptitle('Profils financiers : fraude vs non-fraude', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Ratio dette / revenu
df_full['debt_to_income'] = df_full['total_debt'] / (df_full['yearly_income'] + 1)

dti_fraud = df_full.groupby('fraud_label')['debt_to_income'].median().rename({0:'Non-fraude',1:'Fraude'})
print('Médiane dette/revenu:')
print(dti_fraud.round(3))

In [ ]:
# --- Distance domicile → lieu de transaction (haversine)
# Approximation : on utilise lat/lon de l'utilisateur et on regarde
# si le merchant est ONLINE ou dans un état différent de l'état de résidence
def extract_state_from_address(address):
    """Heuristique : dernier token de l'adresse ~ état US pour cet exemple."""
    return str(address).split(',')[-1].strip() if pd.notna(address) else np.nan

# Approximation simple : transactions dans un état différent de la résidence
# On ne peut pas calculer la vraie distance sans les coordonnées des marchands
# mais on peut flaguer les transactions géographiquement distantes
if 'merchant_state' in df_full.columns:
    df_full['txn_out_of_state'] = (
        df_full['merchant_state'].notna() &
        (df_full['is_online'] == 0) &
        (df_full['merchant_state'] != 'ONLINE')
    ).astype(int)
    # On ne connaît pas le state de résidence depuis users_data directement,
    # mais on peut construire la feature distance haversine avec lat/lon user + zip merchant
    print('Feature `txn_out_of_state` prête comme proxy géographique.')
    print(df_full.groupby('txn_out_of_state')['fraud_label'].agg(['mean','count']))

In [ ]:
# --- Distance haversine (user home lat/lon vs zip centroïde du marchand)
# Note : nécessiterait une table zip → lat/lon. Proxy ici : taux de fraude par user lat/lon bucket
if 'latitude' in df_full.columns:
    df_full['user_lat_bucket'] = pd.cut(df_full['latitude'], bins=10)
    lat_fraud = df_full.groupby('user_lat_bucket')['fraud_label'].mean() * 100
    fig, ax = plt.subplots(figsize=(12, 4))
    lat_fraud.plot.bar(ax=ax, color='#4C72B0')
    ax.set_title('Taux de fraude par zone géographique (latitude domicile)')
    ax.set_ylabel('Taux de fraude (%)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Section 6 — Analyse MCC (Merchant Category Codes)

In [ ]:
# Merge MCC descriptions
df_mcc = df_full.merge(mcc_df, on='mcc', how='left')

# Top MCCs par volume de fraude
mcc_stats = df_mcc.groupby(['mcc', 'mcc_description'])['fraud_label'].agg(
    fraud_count='sum', total='count'
).reset_index()
mcc_stats['fraud_rate'] = mcc_stats['fraud_count'] / mcc_stats['total']
mcc_stats = mcc_stats[mcc_stats['total'] >= 50]  # filtre les catégories trop rares

print('=== Top 15 MCCs par taux de fraude (min 50 transactions) ===')
top_fraud_mcc = mcc_stats.sort_values('fraud_rate', ascending=False).head(15)
print(top_fraud_mcc[['mcc', 'mcc_description', 'fraud_count', 'total', 'fraud_rate']].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
top15 = mcc_stats.sort_values('fraud_rate', ascending=False).head(15)
labels_mcc = [f"{row['mcc']}\n{row['mcc_description'][:25]}" for _, row in top15.iterrows()]
ax.barh(labels_mcc[::-1], top15['fraud_rate'].values[::-1] * 100, color='#DD8452')
ax.set_title('Top 15 catégories marchandes par taux de fraude')
ax.set_xlabel('Taux de fraude (%)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Catégories à surveiller (money transfer, betting, digital goods)
high_risk_mcc = ['4829', '7995', '5815', '5816', '5094']
for code in high_risk_mcc:
    desc = mcc_codes.get(code, 'N/A')
    subset = df_mcc[df_mcc['mcc'] == int(code)]
    if len(subset) > 0:
        rate = subset['fraud_label'].mean() * 100
        n = len(subset)
        print(f'MCC {code} — {desc}: {n} txn, taux fraude = {rate:.3f}%')

## Section 7 — Feature Engineering Recommandé

> **Règle d'or** : toutes les features doivent être calculables **sans connaître le client** (cold-start).  
> Interdiction d'agréger par `client_id` (leakage).


In [ ]:
import pandas as pd
import numpy as np

def build_features(txn_df, cards_df, users_df, mcc_df, fraud_labels=None, fit=True, mcc_fraud_rates=None):
    """
    Construit le feature matrix complet.

    Paramètres
    ----------
    txn_df        : transactions (train ou eval)
    cards_df      : cards_data
    users_df      : users_data
    mcc_df        : mcc codes DataFrame
    fraud_labels  : labels (None pour eval)
    fit           : True = calcul mcc_fraud_rates (train), False = réutilise mcc_fraud_rates (eval)
    mcc_fraud_rates : dict {mcc: fraud_rate} calculé sur le train
    """
    df = txn_df.copy()
    df['amount'] = df['amount'].astype(str).str.replace('$', '', regex=False).astype(float)
    df['date'] = pd.to_datetime(df['date'])

    if fraud_labels is not None:
        df = df.merge(fraud_labels, on='transaction_id', how='left')
        df['fraud_label'] = df['fraud_label'].fillna(0).astype(int)

    # ── Features temporelles ────────────────────────────────────────
    df['hour']        = df['date'].dt.hour
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month']       = df['date'].dt.month
    df['is_night']    = df['hour'].between(0, 5).astype(int)   # 0h–5h
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

    # ── Features montant ────────────────────────────────────────────
    df['log_amount']         = np.log1p(df['amount'].clip(lower=0))
    df['is_negative_amount'] = (df['amount'] < 0).astype(int)
    df['amount_abs']         = df['amount'].abs()

    # ── Features transaction type ────────────────────────────────────
    df['is_online']  = (df['merchant_city'] == 'ONLINE').astype(int)
    df['is_swipe']   = (df['use_chip'] == 'Swipe Transaction').astype(int)
    df['is_chip']    = (df['use_chip'] == 'Chip Transaction').astype(int)

    # ── Features géographiques ───────────────────────────────────────
    us_states = {'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA',
                 'KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
                 'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT',
                 'VA','WA','WV','WI','WY','DC','PR'}
    df['is_foreign'] = (
        df['merchant_state'].notna() &
        ~df['merchant_state'].isin(us_states) &
        (df['is_online'] == 0)
    ).astype(int)

    # ── Features erreurs ────────────────────────────────────────────
    df['has_error'] = df['errors'].notna().astype(int)

    # ── Merge cartes ─────────────────────────────────────────────────
    cards = cards_df.rename(columns={'id': 'card_id'}).copy()
    cards['credit_limit'] = cards['credit_limit'].astype(str).str.replace('$','',regex=False).str.replace(',','',regex=False).astype(float)
    df = df.merge(cards[['card_id','has_chip','credit_limit','card_on_dark_web',
                          'expires','year_pin_last_changed','card_type','card_brand']], on='card_id', how='left')

    df['card_on_dark_web_flag'] = (df['card_on_dark_web'] == 'Yes').astype(int)
    df['amount_to_limit_ratio'] = df['amount_abs'] / (df['credit_limit'].fillna(1) + 1)
    df['chip_mismatch'] = ((df['has_chip'] == 'YES') & (df['use_chip'] == 'Swipe Transaction')).astype(int)

    # Carte expirée
    df['card_expiry_date'] = pd.to_datetime(df['expires'], format='%m/%Y', errors='coerce')
    df['card_expired_at_txn'] = (df['date'] > df['card_expiry_date']).fillna(0).astype(int)

    # Temps depuis changement du PIN (en années)
    df['years_since_pin_change'] = df['date'].dt.year - df['year_pin_last_changed'].fillna(df['date'].dt.year)

    # ── Merge users ───────────────────────────────────────────────────
    users = users_df.rename(columns={'id': 'client_id'}).copy()
    for col in ['per_capita_income', 'yearly_income', 'total_debt']:
        users[col] = users[col].astype(str).str.replace('$','',regex=False).str.replace(',','',regex=False).astype(float)
    df = df.merge(users[['client_id','credit_score','yearly_income','total_debt',
                          'current_age','gender','latitude','longitude']], on='client_id', how='left')

    df['debt_to_income'] = df['total_debt'] / (df['yearly_income'].fillna(1) + 1)

    # ── MCC fraud rate (target encoding par cross-val sur train seulement) ─
    if fit:
        if fraud_labels is not None:
            mcc_fraud_rates = df.groupby('mcc')['fraud_label'].mean().to_dict()
        else:
            mcc_fraud_rates = {}
    global_mean = np.mean(list(mcc_fraud_rates.values())) if mcc_fraud_rates else 0
    df['mcc_fraud_rate'] = df['mcc'].map(mcc_fraud_rates).fillna(global_mean)

    # ── Sélection features finales ────────────────────────────────────
    feature_cols = [
        # Montant
        'log_amount', 'is_negative_amount', 'amount_to_limit_ratio',
        # Temporel
        'hour', 'day_of_week', 'month', 'is_night', 'is_weekend',
        # Type de transaction
        'is_online', 'is_swipe', 'is_chip',
        # Géo
        'is_foreign',
        # Erreur
        'has_error',
        # Carte
        'card_on_dark_web_flag', 'chip_mismatch', 'card_expired_at_txn',
        'years_since_pin_change',
        # User
        'credit_score', 'debt_to_income', 'current_age',
        # MCC
        'mcc_fraud_rate',
        # Catégorielles
        'use_chip', 'merchant_state', 'card_type', 'card_brand', 'gender',
    ]
    feature_cols = [c for c in feature_cols if c in df.columns]

    return df[['transaction_id'] + feature_cols + (['fraud_label'] if fraud_labels is not None else [])], mcc_fraud_rates


# Test sur un échantillon
sample_txn = txn.head(10000)
X_sample, mcc_rates = build_features(sample_txn, cards, users, mcc_df, fraud_labels=labels)
print(f'Features construites: {X_sample.shape}')
print(X_sample.dtypes)

### Récapitulatif des features recommandées

| Feature | Source | Rationale |
|---|---|---|
| `log_amount` | transaction | Normalise la distribution fortement asymétrique |
| `is_negative_amount` | transaction | Remboursements suspects / retournements |
| `amount_to_limit_ratio` | txn + card | Dépasser la limite habituelle est un signal |
| `hour`, `day_of_week`, `month` | date | Patterns temporels de fraude |
| `is_night`, `is_weekend` | date | Fraude concentrée hors heures ouvrables |
| `is_online` | merchant_city | Transactions en ligne = moins de vérification physique |
| `is_swipe` / `is_chip` | use_chip | Mode de paiement lié au risque |
| `is_foreign` | merchant_state | Transaction hors États-Unis |
| `has_error` | errors | Erreur lors de la transaction = signal |
| `card_on_dark_web_flag` | cards | **Signal très fort** : carte compromise |
| `chip_mismatch` | card + txn | Swipe sur une carte chip = suspect |
| `card_expired_at_txn` | card + date | Utilisation d'une carte expirée |
| `years_since_pin_change` | card + date | PIN récemment changé = activité suspecte |
| `credit_score` | users | Profil de risque de l'utilisateur |
| `debt_to_income` | users | Fragilité financière |
| `current_age` | users | Corrélation démographique |
| `mcc_fraud_rate` | mcc + labels | Certaines catégories marchandes sont très à risque |

**Features à ÉVITER (leakage cold-start)** :
- Toute agrégation par `client_id` : rolling fraud rate, nombre de transactions passées, etc.
- Ces features ne seront pas calculables pour les clients inconnus en évaluation.

## Section 8 — Modèles Conseillés

**Contexte** : classification binaire, déséquilibre extrême (~0.15%), généralisation cold-start (nouveaux clients).

### Recommandations par ordre de priorité

---

#### 1. LightGBM — Recommandé en priorité
```python
import lightgbm as lgb

params = {
    'objective': 'binary',
    'metric': 'auc',
    'is_unbalance': True,          # gère le déséquilibre automatiquement
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
}
# Avantages :
# - Gère nativement les catégorielles (use_chip, merchant_state, mcc)
# - Très rapide sur 200k lignes
# - is_unbalance=True adapté au ~0.15% fraude
# - Robuste aux valeurs manquantes
```

---

#### 2. XGBoost
```python
import xgboost as xgb

# scale_pos_weight = nb_non_fraudes / nb_fraudes (~666)
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'scale_pos_weight': 666,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
}
# Avantages :
# - Complémentaire à LightGBM pour un ensemble final
# - scale_pos_weight calibré sur le vrai ratio imbalance
```

---

#### 3. CatBoost
```python
from catboost import CatBoostClassifier

cat_features = ['use_chip', 'merchant_state', 'card_type', 'card_brand', 'gender']
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    cat_features=cat_features,
    eval_metric='AUC',
)
# Avantages :
# - Encodage catégoriel natif sans preprocessing
# - auto_class_weights='Balanced' pour l'imbalance
# - Résistant à l'overfitting sur peu de fraudes
```

---

#### 4. Random Forest (baseline robuste)
```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced_subsample',  # rééquilibrage par arbre
    max_depth=None,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
# Avantages :
# - Interprétable (feature importance)
# - Pas de tuning délicat
# - Bonne baseline
```

---

#### 5. Isolation Forest (détection d'anomalies, non-supervisé)
```python
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination=0.0015,   # taux de fraude estimé
    n_estimators=200,
    random_state=42,
)
# Avantages :
# - Ne dépend pas du tout des labels → parfait pour cold-start
# - Complémentaire en ensemble : score anomalie comme feature pour LightGBM
# - Utile si peu de labels disponibles
```

---

#### 6. (Optionnel) Réseau de neurones avec Focal Loss
```python
# Focal Loss = Binary Cross-Entropy qui down-sample les cas faciles
# Conçu pour l'imbalance extrême (papers VISA/Mastercard fraud detection)
# alpha contrôle le poids des positifs, gamma contrôle le focus
# focal_loss = -alpha * (1 - p)^gamma * log(p)
#
# Implémentation : torch + tabnet, ou keras
# Recommandé si LightGBM plafonne
```

---

### Stratégie de validation

```python
from sklearn.model_selection import StratifiedKFold

# StratifiedKFold pour préserver le ratio de fraude dans chaque fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Métriques à suivre :
# - ROC-AUC (principale)
# - Precision-Recall AUC (plus pertinent que ROC sur déséquilibre extrême)
# - F1-score avec threshold optimisé (pas forcément 0.5)
```

### Résumé des choix

| Modèle | Priorité | Imbalance | Catégorielles | Cold-start |
|---|---|---|---|---|
| LightGBM | ⭐⭐⭐ | `is_unbalance` | Natif | Oui |
| XGBoost | ⭐⭐⭐ | `scale_pos_weight` | Encoding manuel | Oui |
| CatBoost | ⭐⭐ | `auto_class_weights` | Natif | Oui |
| Random Forest | ⭐⭐ | `balanced_subsample` | Encoding manuel | Oui |
| Isolation Forest | ⭐ (feature) | Contamination | - | Excellent |
| NN + Focal Loss | ⭐ (avancé) | Focal Loss | Embedding | Oui |